# F1 Dataset — Exploratory Data Analysis
**Datum:** Dan 1 — Data Engineering Internship  
**Dataset:** dataEngineeringDataset.csv  
**Autor:** Andrej Vukoje

## 1. Loading the Dataset
Loading the complete CSV file and inspecting its basic characteristics.

In [1]:
import pandas as pd
import numpy as np

# Load the full dataset
path = r"C:\Users\andrej.vukoje\Downloads\DE (1)\DE\dataEngineeringDataset.csv"
df = pd.read_csv(path)

print(f"Total rows:    {len(df):,}")
print(f"Total columns: {len(df.columns)}")
print(f"Size in MB:    {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

C:\Users\andrej.vukoje\AppData\Local\Temp\ipykernel_8644\2508969840.py:6: DtypeWarning: Columns (0: position, 1: positionText, 2: milliseconds, 3: fastestLap, 4: fastestLapSpeed, 5: alt, 6: number_drivers, 7: duration) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Total rows:    518,417
Total columns: 76


Size in MB:    1346.9 MB


## 2. Data Types
Pandas detected mixed types in some columns (DtypeWarning) —
the `position` column contains both integers and strings such as "R" (retired), "D", "W".  
This will need to be handled during the transformation phase.

In [2]:
print(df.dtypes)

Unnamed: 0                             int64
resultId                               int64
raceId                                 int64
driverId                               int64
constructorId                          int64
                                      ...   
points_constructorstandings          float64
position_constructorstandings          int64
positionText_constructorstandings      int64
wins_constructorstandings              int64
status                                   str
Length: 76, dtype: object


## 3. Null Values
Data quality check — identifying missing values across all columns.

In [3]:

nulls = df.isnull().sum()
nulls = nulls[nulls > 0].sort_values(ascending=False)

print("Columns with null values:")
print(nulls)

Columns with null values:
Series([], dtype: int64)


## 4. Dataset Statistics

**Key findings:**
- Dataset covers the period **2012—2023** (12 seasons of modern F1)
- The 2023 season is **incomplete** — only 12 out of 23 races are present
- 65 drivers, 20 constructors, 34 circuits across the entire period

In [4]:
print("=== DATASET STATISTICS ===")
print(f"Number of races (raceId):         {df['raceId'].nunique()}")
print(f"Number of drivers (driverId):     {df['driverId'].nunique()}")
print(f"Number of teams (constructorId):  {df['constructorId'].nunique()}")
print(f"Number of circuits (circuitId):   {df['circuitId'].nunique()}")
print(f"Number of statuses (statusId):    {df['statusId'].nunique()}")
print(f"Year range:                       {df['year'].min()} — {df['year'].max()}")
print(f"Number of seasons:                {df['year'].nunique()}")

=== DATASET STATISTICS ===
Number of races (raceId):         232
Number of drivers (driverId):     65
Number of teams (constructorId):  20
Number of circuits (circuitId):   34
Number of statuses (statusId):    65
Year range:                       2012 — 2023
Number of seasons:                12


## 5. Granularity Check
Expected granularity: **one row = one lap per driver per race**.

In [5]:
total_rows = len(df)
unique_combinations = df[['raceId', 'driverId', 'lap']].drop_duplicates()

print(f"Total rows:                       {total_rows:,}")
print(f"Unique raceId+driverId+lap combos: {len(unique_combinations):,}")

if total_rows == len(unique_combinations):
    print("✓ Granularity OK — each row is a unique lap")
else:
    print("⚠ Duplicates detected — further investigation needed")

Total rows:                       518,417
Unique raceId+driverId+lap combos: 256,836
⚠ Duplicates detected — further investigation needed


### Finding: Cross-join Issue

The original file contains **twice as many rows** as expected.  
Cause: pit stop data was joined in a way that assigned one row per lap  
for **each pit stop that occurred during the entire race**.

**ETL implications:**
- `FACT_LAP_TIMES` → deduplicate on `['raceId', 'driverId', 'lap']`
- `FACT_PIT_STOPS` → deduplicate on `['raceId', 'driverId', 'stop']`

In [6]:
# Investigate duplicate rows — identify which columns differ
duplicates = df[df.duplicated(subset=['raceId', 'driverId', 'lap'], keep=False)]
duplicates_sample = duplicates.sort_values(['raceId', 'driverId', 'lap']).head(20)

# Display relevant columns to understand the cause of duplicates
display(duplicates_sample[['raceId', 'driverId', 'lap', 'stop', 'lap_pitstops', 'milliseconds_laptimes', 'duration']].head(20))

,raceId,driverId,lap,stop,lap_pitstops,milliseconds_laptimes,duration
116,860,1,1,1,17,100622,22.862
117,860,1,1,2,36,100622,23.464
118,860,1,2,1,17,94297,22.862
119,860,1,2,2,36,94297,23.464
120,860,1,3,1,17,93566,22.862
121,860,1,3,2,36,93566,23.464
122,860,1,4,1,17,93347,22.862
123,860,1,4,2,36,93347,23.464
124,860,1,5,1,17,93446,22.862
125,860,1,5,2,36,93446,23.464


## 6. Race Results Analysis
Top 10 drivers by number of wins in the period 2012—2023.

In [7]:
wins = (
    df[df['positionOrder'] == 1]
    .drop_duplicates(subset=['resultId'])
    [['forename', 'surname', 'resultId']]
    .assign(ime=lambda x: x['forename'] + ' ' + x['surname'])
    .groupby('ime')['resultId']
    .count()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
wins.columns = ['Driver', 'Wins']
display(wins)

,Driver,Wins
0,Lewis Hamilton,86
1,Max Verstappen,44
2,Sebastian Vettel,32
3,Nico Rosberg,23
4,Valtteri Bottas,10
5,Daniel Ricciardo,8
6,Sergio Pérez,6
7,Charles Leclerc,5
8,Fernando Alonso,5
9,Jenson Button,3


## 7. Races per Season

In [8]:
races_per_season = (
    df.drop_duplicates(subset=['raceId'])
    .groupby('year')['raceId']
    .count()
    .reset_index()
)
races_per_season.columns = ['Year', 'Number of races']
display(races_per_season)

,Year,Number of races
0,2012,20
1,2013,19
2,2014,19
3,2015,19
4,2016,21
5,2017,20
6,2018,21
7,2019,21
8,2020,17
9,2021,21


## 8. Status Distribution
How races end — how many drivers finish, how many retire and for what reasons.

In [9]:
statuses = (
    df.drop_duplicates(subset=['resultId'])
    .groupby('status')['resultId']
    .count()
    .sort_values(ascending=False)
    .head(15)
    .reset_index()
)
statuses.columns = ['Status', 'Count']
display(statuses)

,Status,Count
0,Finished,2472
1,+1 Lap,1196
2,+2 Laps,241
3,Collision,80
4,Accident,50
5,Engine,50
6,Retired,41
7,Brakes,38
8,+3 Laps,37
9,Gearbox,33


## 9. EDA Summary

| Category | Value |
|---|---|
| Total rows | 518,417 |
| Total columns | 76 |
| Null values | 0 |
| Year range | 2012—2023 |
| Number of races | 232 |
| Number of drivers | 65 |
| Number of constructors | 20 |
| Number of circuits | 34 |

**Issues identified in the dataset:**
1. Column `position` has mixed types — strings and integers
2. The 2023 season is incomplete — only 12 out of 23 races present
3. Cross-join duplicates caused by pit stop join — granularity is not clean

**Star schema — dimension tables:**
- `DIM_DRIVER` (65 rows)
- `DIM_CONSTRUCTOR` (20 rows)
- `DIM_CIRCUIT` (34 rows)
- `DIM_RACE` (232 rows)
- `DIM_STATUS` (65 rows)
- `DIM_DATE` — derived from all date columns in DIM_RACE and DIM_DRIVER.dob

**Star schema — fact tables:**
- `FACT_RESULTS` (4,502 rows)
- `FACT_LAP_TIMES` (256,836 rows)
- `FACT_PIT_STOPS` (8,975 rows)
- `FACT_DRIVER_STANDINGS` (4,502 rows)
- `FACT_CONSTRUCTOR_STANDINGS` (2,398 rows)

**Next step:** ETL transformation — splitting into star schema tables

## 10. Schema Validation

Verifying that the star schema extracted from the dataset
matches the expected structure before proceeding to ETL.

In [10]:
# ============================================================
# SCHEMA VALIDATION — verify star schema against actual data
# ============================================================

# DIM_DRIVER — expected 65 unique drivers
dim_driver = df[['driverId', 'driverRef', 'number_drivers', 'code', 
                  'forename', 'surname', 'dob', 'nationality']].drop_duplicates(subset=['driverId'])
print(f"DIM_DRIVER rows: {len(dim_driver)} (expected 65)")

# DIM_CONSTRUCTOR — expected 20
dim_constructor = df[['constructorId', 'constructorRef', 'name', 
                       'nationality_constructors']].drop_duplicates(subset=['constructorId'])
print(f"DIM_CONSTRUCTOR rows: {len(dim_constructor)} (expected 20)")

# DIM_CIRCUIT — expected 34
dim_circuit = df[['circuitId', 'circuitRef', 'name_y', 'location', 
                   'country', 'lat', 'lng', 'alt']].drop_duplicates(subset=['circuitId'])
print(f"DIM_CIRCUIT rows: {len(dim_circuit)} (expected 34)")

# DIM_RACE — expected 232
dim_race = df[['raceId', 'year', 'round', 'circuitId', 'name_x', 'date', 
               'time_races', 'fp1_date', 'fp1_time', 'fp2_date', 'fp2_time',
               'fp3_date', 'fp3_time', 'quali_date', 'quali_time', 
               'sprint_date', 'sprint_time']].drop_duplicates(subset=['raceId'])
print(f"DIM_RACE rows: {len(dim_race)} (expected 232)")

# DIM_STATUS — expected 65
dim_status = df[['statusId', 'status']].drop_duplicates(subset=['statusId'])
print(f"DIM_STATUS rows: {len(dim_status)} (expected 65)")

DIM_DRIVER rows: 65 (expected 65)
DIM_CONSTRUCTOR rows: 20 (expected 20)
DIM_CIRCUIT rows: 34 (expected 34)
DIM_RACE rows: 232 (expected 232)
DIM_STATUS rows: 65 (expected 65)


In [11]:
# FACT tables validation
# FACT_RESULTS — one row per driver per race
fact_results = df[['resultId', 'raceId', 'driverId', 'constructorId', 'statusId',
                    'grid', 'position', 'positionText', 'positionOrder', 'points', 
                    'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 
                    'fastestLapTime', 'fastestLapSpeed']].drop_duplicates(subset=['resultId'])
print(f"FACT_RESULTS rows: {len(fact_results)}")
print(f"  Unique resultId check: {fact_results['resultId'].nunique() == len(fact_results)}")

# FACT_LAP_TIMES — one row per lap per driver per race
fact_lap_times = df[['raceId', 'driverId', 'lap', 'position_laptimes', 
                      'time_laptimes', 'milliseconds_laptimes']].drop_duplicates(subset=['raceId', 'driverId', 'lap'])
print(f"FACT_LAP_TIMES rows: {len(fact_lap_times)}")

# FACT_PIT_STOPS — one row per pit stop per driver per race
fact_pit_stops = df[['raceId', 'driverId', 'stop', 'lap_pitstops', 
                      'time_pitstops', 'duration', 'milliseconds_pitstops']].drop_duplicates(subset=['raceId', 'driverId', 'stop'])
print(f"FACT_PIT_STOPS rows: {len(fact_pit_stops)}")

# FACT_DRIVER_STANDINGS — one row per driver per race
fact_driver_standings = df[['driverStandingsId', 'raceId', 'driverId', 'points_driverstandings',
                             'position_driverstandings', 'positionText_driverstandings', 
                             'wins']].drop_duplicates(subset=['driverStandingsId'])
print(f"FACT_DRIVER_STANDINGS rows: {len(fact_driver_standings)}")
print(f"  Unique driverStandingsId check: {fact_driver_standings['driverStandingsId'].nunique() == len(fact_driver_standings)}")

# FACT_CONSTRUCTOR_STANDINGS
fact_constructor_standings = df[['constructorStandingsId', 'raceId', 'constructorId', 
                                  'points_constructorstandings', 'position_constructorstandings',
                                  'positionText_constructorstandings', 
                                  'wins_constructorstandings']].drop_duplicates(subset=['constructorStandingsId'])
print(f"FACT_CONSTRUCTOR_STANDINGS rows: {len(fact_constructor_standings)}")
print(f"  Unique constructorStandingsId check: {fact_constructor_standings['constructorStandingsId'].nunique() == len(fact_constructor_standings)}")

FACT_RESULTS rows: 4502
  Unique resultId check: True
FACT_LAP_TIMES rows: 256836
FACT_PIT_STOPS rows: 8975
FACT_DRIVER_STANDINGS rows: 4502
  Unique driverStandingsId check: True
FACT_CONSTRUCTOR_STANDINGS rows: 2398
  Unique constructorStandingsId check: True


In [12]:
# REFERENTIAL INTEGRITY — do all FK values exist in their dimension?
print("=== REFERENTIAL INTEGRITY CHECKS ===")

checks = [
    ("FACT_RESULTS → DIM_RACE",        fact_results['raceId'].isin(dim_race['raceId']).all()),
    ("FACT_RESULTS → DIM_DRIVER",      fact_results['driverId'].isin(dim_driver['driverId']).all()),
    ("FACT_RESULTS → DIM_CONSTRUCTOR", fact_results['constructorId'].isin(dim_constructor['constructorId']).all()),
    ("FACT_RESULTS → DIM_STATUS",      fact_results['statusId'].isin(dim_status['statusId']).all()),
    ("FACT_LAP_TIMES → DIM_RACE",      fact_lap_times['raceId'].isin(dim_race['raceId']).all()),
    ("FACT_LAP_TIMES → DIM_DRIVER",    fact_lap_times['driverId'].isin(dim_driver['driverId']).all()),
    ("FACT_PIT_STOPS → DIM_RACE",      fact_pit_stops['raceId'].isin(dim_race['raceId']).all()),
    ("FACT_PIT_STOPS → DIM_DRIVER",    fact_pit_stops['driverId'].isin(dim_driver['driverId']).all()),
    ("DIM_RACE → DIM_CIRCUIT",         dim_race['circuitId'].isin(dim_circuit['circuitId']).all()),
]

for name, result in checks:
    status = "✓" if result else "✗ FAILED"
    print(f"  {status}  {name}")

=== REFERENTIAL INTEGRITY CHECKS ===
  ✓  FACT_RESULTS → DIM_RACE
  ✓  FACT_RESULTS → DIM_DRIVER
  ✓  FACT_RESULTS → DIM_CONSTRUCTOR
  ✓  FACT_RESULTS → DIM_STATUS
  ✓  FACT_LAP_TIMES → DIM_RACE
  ✓  FACT_LAP_TIMES → DIM_DRIVER
  ✓  FACT_PIT_STOPS → DIM_RACE
  ✓  FACT_PIT_STOPS → DIM_DRIVER
  ✓  DIM_RACE → DIM_CIRCUIT


### Results
- All dimension tables match expected row counts
- All foreign key relationships are intact — no orphaned records
- Granularity confirmed: FACT_LAP_TIMES deduplicates correctly to 256,836 rows
- Cross-join issue identified and handled via deduplication strategy

**Conclusion: Schema is valid. ETL can proceed.**

## 11. DIM_DATE — Date Dimension

A date dimension is a standard star schema component.  
Instead of storing raw date strings in fact/dimension tables, all date columns reference a single `DIM_DATE` by a surrogate integer key (`date_id`, e.g. `20230305`).

**Columns sourced from `DIM_RACE`:**
- `date` — race day
- `fp1_date`, `fp2_date`, `fp3_date` — practice sessions
- `quali_date` — qualifying
- `sprint_date` — sprint race (where applicable)

**Also used by:** `DIM_DRIVER.dob` (date of birth)

In [13]:
import pandas as pd

# Collect all unique dates from every date column in DIM_RACE + driver DOB
date_columns = ['date', 'fp1_date', 'fp2_date', 'fp3_date', 'quali_date', 'sprint_date']

all_dates = pd.concat([
    pd.to_datetime(dim_race[col], errors='coerce') for col in date_columns
] + [
    pd.to_datetime(dim_driver['dob'], errors='coerce')
]).dropna().drop_duplicates().sort_values().reset_index(drop=True)

dim_date = pd.DataFrame({'date': all_dates})
dim_date['date_id']     = dim_date['date'].dt.strftime('%Y%m%d').astype(int)
dim_date['year']        = dim_date['date'].dt.year
dim_date['month']       = dim_date['date'].dt.month
dim_date['month_name']  = dim_date['date'].dt.strftime('%B')
dim_date['quarter']     = dim_date['date'].dt.quarter
dim_date['day_of_week'] = dim_date['date'].dt.dayofweek   # 0 = Monday
dim_date['day_name']    = dim_date['date'].dt.strftime('%A')
dim_date['is_weekend']  = dim_date['date'].dt.dayofweek >= 5

print(f"DIM_DATE rows: {len(dim_date)}")
print(f"Date range:    {dim_date['date'].min().date()} — {dim_date['date'].max().date()}")
display(dim_date.head(10))

DIM_DATE rows: 407
Date range:    1969-01-03 — 2023-07-30


C:\Users\andrej.vukoje\AppData\Local\Temp\ipykernel_8644\579359456.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(dim_race[col], errors='coerce') for col in date_columns
C:\Users\andrej.vukoje\AppData\Local\Temp\ipykernel_8644\579359456.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(dim_race[col], errors='coerce') for col in date_columns
C:\Users\andrej.vukoje\AppData\Local\Temp\ipykernel_8644\579359456.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(dim_race[col], errors='coerce') for col in date_columns
C:\Users\a

,date,date_id,year,month,month_name,quarter,day_of_week,day_name,is_weekend
0,1969-01-03,19690103,1969,1,January,1,4,Friday,False
1,1971-02-24,19710224,1971,2,February,1,2,Wednesday,False
2,1976-08-27,19760827,1976,8,August,3,4,Friday,False
3,1977-01-14,19770114,1977,1,January,1,4,Friday,False
4,1979-10-17,19791017,1979,10,October,4,2,Wednesday,False
5,1980-01-19,19800119,1980,1,January,1,5,Saturday,True
6,1981-04-25,19810425,1981,4,April,2,5,Saturday,True
7,1981-07-29,19810729,1981,7,July,3,2,Wednesday,False
8,1981-10-19,19811019,1981,10,October,4,0,Monday,False
9,1982-03-18,19820318,1982,3,March,1,3,Thursday,False


In [14]:
# Referential integrity — DIM_RACE date columns → DIM_DATE
print("=== DIM_DATE REFERENTIAL INTEGRITY ===")

date_ids = set(dim_date['date_id'])

def check_date_col(label, series):
    parsed = pd.to_datetime(series, errors='coerce').dropna()
    keys = parsed.dt.strftime('%Y%m%d').astype(int)
    ok = keys.isin(date_ids).all()
    status = "✓" if ok else "✗ FAILED"
    print(f"  {status}  {label}")

check_date_col("DIM_RACE.date      → DIM_DATE", dim_race['date'])
check_date_col("DIM_RACE.fp1_date  → DIM_DATE", dim_race['fp1_date'])
check_date_col("DIM_RACE.fp2_date  → DIM_DATE", dim_race['fp2_date'])
check_date_col("DIM_RACE.fp3_date  → DIM_DATE", dim_race['fp3_date'])
check_date_col("DIM_RACE.quali_date→ DIM_DATE", dim_race['quali_date'])
check_date_col("DIM_RACE.sprint_date→DIM_DATE", dim_race['sprint_date'])
check_date_col("DIM_DRIVER.dob     → DIM_DATE", dim_driver['dob'])

=== DIM_DATE REFERENTIAL INTEGRITY ===
  ✓  DIM_RACE.date      → DIM_DATE
  ✓  DIM_RACE.fp1_date  → DIM_DATE
  ✓  DIM_RACE.fp2_date  → DIM_DATE
  ✓  DIM_RACE.fp3_date  → DIM_DATE
  ✓  DIM_RACE.quali_date→ DIM_DATE
  ✓  DIM_RACE.sprint_date→DIM_DATE
  ✓  DIM_DRIVER.dob     → DIM_DATE


C:\Users\andrej.vukoje\AppData\Local\Temp\ipykernel_8644\1145062436.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series, errors='coerce').dropna()
C:\Users\andrej.vukoje\AppData\Local\Temp\ipykernel_8644\1145062436.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series, errors='coerce').dropna()
C:\Users\andrej.vukoje\AppData\Local\Temp\ipykernel_8644\1145062436.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series, errors='coerce').dropna()
C:\Users\andrej.vukoje\AppData\Local\Temp\ipyk